# DQN で CartPole を解く(演習)

解説資料は `research-handbook/reinforcement-learning/dqn.md`。
このnotebookは**直接編集せず**、自分の卒研repoにコピーして使うこと(手順は `technical-handbook/colab/use-github-repo.md`)。PyTorchを使うためColab推奨。

「---- ここから課題 ----」の区間を埋めながら上から順に実行する。
解答付き版は `research-handbook/notebooks/dqn_cartpole_solution.ipynb` にあるが、まず自力で取り組むこと。

テーブルQ学習(`cliff-walking.md`)の $Q$ テーブルをニューラルネットに置き換え、DQNの2つの工夫を実装する。

- **experience replay**: 遷移をバッファに貯めてランダムに取り出す → データの時間相関を壊す
- **target network**: TD目標を作るネットワークを一定期間固定する → 「動く的」を止める

In [ ]:
# Colab の場合は初回のみ実行してください
# !pip install -q gymnasium torch
import random
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env = gym.make("CartPole-v1")
obs_dim, n_actions = 4, 2

## ネットワークと最適化器

$Q_\theta(s, \cdot)$ を出力する小さなMLPを2つ用意します。`q_net`(学習する側)と `target_net`(目標を作る側)です。

In [ ]:
class QNet(nn.Module):
    """Q(s,·) を出力する小さな全結合ネットワーク"""
    def __init__(self, obs_dim, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)

q_net = QNet(obs_dim, n_actions).to(device)
target_net = QNet(obs_dim, n_actions).to(device)
target_net.load_state_dict(q_net.state_dict())   # 最初は同じ重みからスタート
optimizer = optim.Adam(q_net.parameters(), lr=1e-3)

## 課題1: Replay Buffer

`deque(maxlen=...)` によるリングバッファです。**一様ランダムにサンプル**することが本質で、直近の遷移だけで学習すると強い時間相関のためネットワークが壊れやすくなります(`dqn.md` 参照)。

In [ ]:
class ReplayBuffer:
    """課題1: experience replay 用のリングバッファ"""
    def __init__(self, capacity=50_000):
        self.buf = deque(maxlen=capacity)

    def push(self, s, a, r, s_next, done):
        self.buf.append((s, a, r, s_next, done))

    def sample(self, batch_size):
        # ---- ここから課題1 ----
        # バッファから batch_size 個の遷移を「一様ランダムに」取り出す
        # (直近の遷移だけで学習すると何がまずいか、dqn.md で復習すること)
        batch = None
        # ---- ここまで ----
        s, a, r, s_next, done = map(np.array, zip(*batch))
        to = lambda x, dt: torch.as_tensor(x, dtype=dt, device=device)
        return (to(s, torch.float32), to(a, torch.int64), to(r, torch.float32),
                to(s_next, torch.float32), to(done, torch.float32))

    def __len__(self):
        return len(self.buf)

## 課題2: TD目標と損失

$$L(\theta) = \mathbb{E}\left[ \left( r + \gamma (1 - d) \max_{a'} Q_{\theta^-}(s', a') - Q_\theta(s, a) \right)^2 \right]$$

実装のポイントは2つ。(1) 目標側は `torch.no_grad()` で勾配を止める。(2) 終端(`done`)では将来価値の項を0にする。CartPoleでは**時間切れ(truncated)は失敗ではない**ため、`done` には `terminated` のみを入れることに注意してください(ここを間違えると性能が大きく落ちます)。

In [ ]:
def train_step(buffer, batch_size=64, gamma=0.99):
    """課題2: TD目標の計算と1回の勾配更新"""
    if len(buffer) < batch_size:
        return None
    s, a, r, s_next, done = buffer.sample(batch_size)

    # ---- ここから課題2 ----
    # (1) q_sa: q_net(s) から「実際に選んだ行動 a」の値だけを取り出す (gather を使う)
    # (2) target: r + gamma * (1 - done) * max_a' Q_target(s', a')
    #     目標側は torch.no_grad() で勾配を止めること (どちらのネットを使うかに注意)
    q_sa = None
    target = None
    # ---- ここまで ----

    loss = nn.functional.mse_loss(q_sa, target)
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(q_net.parameters(), 10.0)   # 勾配クリップで安定化
    optimizer.step()
    return loss.item()

## 課題3: 学習ループ

εを1.0から0.05まで線形に減らしながら(アニーリング)、ステップごとに勾配更新、500ステップごとにtarget networkを同期します。

In [ ]:
def epsilon_by_step(step, eps_start=1.0, eps_end=0.05, decay_steps=5_000):
    """課題3: εを eps_start から eps_end まで decay_steps かけて線形に減らす"""
    # ---- ここから課題3 ----
    # step >= decay_steps では eps_end に張り付くようにすること

    # ---- ここまで ----

def select_action(s, eps, rng):
    if rng.random() < eps:
        return rng.integers(n_actions)           # 探索
    with torch.no_grad():
        q = q_net(torch.as_tensor(s, dtype=torch.float32, device=device))
    return int(q.argmax().item())                # 活用

buffer = ReplayBuffer()
rng = np.random.default_rng(0)
TARGET_UPDATE_EVERY = 500    # 何ステップごとに target network を同期するか
history, losses = [], []
global_step = 0

for ep in range(300):
    s, _ = env.reset(seed=int(rng.integers(1_000_000)))
    ep_return, done = 0.0, False
    while not done:
        eps = epsilon_by_step(global_step)
        a = select_action(s, eps, rng)
        s_next, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        buffer.push(s, a, r, s_next, float(terminated))  # 注意: 打ち切り(truncated)は「失敗」ではない
        loss = train_step(buffer)
        if loss is not None:
            losses.append(loss)
        if global_step % TARGET_UPDATE_EVERY == 0:
            target_net.load_state_dict(q_net.state_dict())
        s = s_next
        ep_return += r
        global_step += 1
    history.append(ep_return)
    if (ep + 1) % 50 == 0:
        print(f"episode {ep+1}: 直近50ep平均リターン = {np.mean(history[-50:]):.1f}, eps={eps:.2f}")

In [ ]:
def moving_average(x, w=20):
    return np.convolve(x, np.ones(w) / w, mode="valid")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history, alpha=0.3)
axes[0].plot(moving_average(history))
axes[0].set_xlabel("episode"); axes[0].set_ylabel("return"); axes[0].set_title("Episode return")
axes[1].plot(moving_average(losses, 100))
axes[1].set_xlabel("gradient step"); axes[1].set_ylabel("TD loss"); axes[1].set_title("Loss")
plt.show()

## 発展課題

1. `TARGET_UPDATE_EVERY` を 1(毎ステップ同期 = target networkなしとほぼ同じ)にすると何が起きるか
2. replay buffer の容量を 500 に減らすと何が起きるか
3. TD目標の `max` を Double DQN(行動選択は `q_net`、評価は `target_net`)に置き換えて比較する(`dqn.md` 6節)
4. すべての比較は**複数seedの平均**で行うこと(`hyperparameter.md`)

## まとめ

- DQN = Q学習 + 関数近似 + experience replay + target network
- deadly triad(関数近似・ブートストラップ・off-policy)を2つの工夫で抑え込んでいる
- ハイパーパラメータに敏感なので、変更は1つずつ・複数seedで確認する
- 連続行動への拡張は `soft-actor-critic.md`、on-policy側の発展は `ppo-mppi.md` へ